In [6]:
!pip install xgboost 

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
    --------------------------------------- 1.3/101.7 MB 2.8 MB/s eta 0:00:36
    --------------------------------------- 1.8/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 3.1/101.7 MB 4.2 MB/s eta 0:00:24
   - -------------------------------------- 3.9/101.7 MB 4.1 MB/s eta 0:00:25
   - -------------------------------------- 3.9/101.7 MB 4.1 MB/s eta 0:00:25
   - -------------------------------------- 4.2/101.7 MB 3.0 MB/s eta 0:00:33
   - -------------------------------------- 4.7/101.7 MB 2.7 MB/s eta 0:00:36
   -- ------------------------------------- 5.2/101.7 MB 2.7 MB/s eta 0:00:37
   -- ------------------------------------- 5.8/101.7 MB 2.7 MB/s eta 0:00:36
   -- --------


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import joblib
import pandas as pd
import numpy as np
import os

# Load Model Files

model = joblib.load("../ML/models/model.pkl")
scaler = joblib.load("../ML/models/scaler.pkl")
encoders = joblib.load("../ML/models/encoders.pkl")
feature_names = joblib.load("../ML/models/features.pkl")

print("MODEL LOADED")

c:\Users\Asad Ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Asad Ali\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


MODEL LOADED


In [2]:
from scapy.all import sniff
from scapy.layers.inet import IP, TCP, UDP

In [5]:
def extract_features(packet):

    features = {}

    # Default values
    for feature in feature_names:
        features[feature] = 0

    try:

        # Duration
        features['duration'] = 0

        # Protocol
        if packet.haslayer(TCP):
            proto = 'tcp'
        elif packet.haslayer(UDP):
            proto = 'udp'
        else:
            proto = 'icmp'

        # Encode protocol
        features['protocol_type'] = (
            encoders['protocol_type']
            .transform([proto])[0]
            if proto in encoders['protocol_type'].classes_
            else 0
        )

        # Service
        service = 'http'

        features['service'] = (
            encoders['service']
            .transform([service])[0]
            if service in encoders['service'].classes_
            else 0
        )

        # Flag
        flag = 'SF'

        features['flag'] = (
            encoders['flag']
            .transform([flag])[0]
            if flag in encoders['flag'].classes_
            else 0
        )

        # Bytes
        features['src_bytes'] = len(packet)

        features['dst_bytes'] = 0

        return pd.DataFrame([features])

    except Exception as e:
        print("Feature Extraction Error:", e)
        return None

In [6]:
def process_pack(packet):

    data = extract_features(packet)

    if data is not None:

        # Reorder Columns
        data = data[feature_names]

        # Scale
        data_scaled = scaler.transform(data)

        # Predict
        prediction = model.predict(data_scaled)[0]

        # Probability
        probability = model.predict_proba(data_scaled)[0]

        if prediction == 1:
            print("\n⚠️ ATTACK DETECTED")
            print("Confidence:", np.max(probability))

        else:
            print("Normal Traffic")

In [7]:
print("LIVE IDS STARTED...")

sniff(
    prn=process_pack,
    store=False
)

LIVE IDS STARTED...

⚠️ ATTACK DETECTED
Confidence: 0.99830085

⚠️ ATTACK DETECTED
Confidence: 0.9648798

⚠️ ATTACK DETECTED
Confidence: 0.9648798

⚠️ ATTACK DETECTED
Confidence: 0.9969663

⚠️ ATTACK DETECTED
Confidence: 0.9970445

⚠️ ATTACK DETECTED
Confidence: 0.9970445

⚠️ ATTACK DETECTED
Confidence: 0.99830085

⚠️ ATTACK DETECTED
Confidence: 0.9969663

⚠️ ATTACK DETECTED
Confidence: 0.9970445

⚠️ ATTACK DETECTED
Confidence: 0.99759656

⚠️ ATTACK DETECTED
Confidence: 0.99759656

⚠️ ATTACK DETECTED
Confidence: 0.9970445

⚠️ ATTACK DETECTED
Confidence: 0.9958234

⚠️ ATTACK DETECTED
Confidence: 0.9958234

⚠️ ATTACK DETECTED
Confidence: 0.9958234

⚠️ ATTACK DETECTED
Confidence: 0.99765944

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.9590278

⚠️ ATTACK DETECTED
Confidence: 0.98962575

⚠️ ATTA

<Sniffed: TCP:0 UDP:0 ICMP:0 Other:0>